In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(ROOT))

print("Añadido al path:", ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

pd.set_option("display.max_columns", None)

In [ ]:
import io
import os
import matplotlib.pyplot as plt

from src.utils.funciones_minio import crear_cliente_minio, bajar_minio
from src.config import PATH_PRIMARIOS_LIMPIO

OBJ_VIVIENDAS_VENTA = "viviendas_venta.parquet"
OBJ_VIVIENDAS_ALQUILER = "viviendas_alquiler.parquet"


In [ ]:
client = crear_cliente_minio()

In [ ]:
df_venta = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_VENTA)
if not isinstance(df_venta, pd.DataFrame):
        df_venta = pd.read_parquet(io.BytesIO(df_venta))
df_alquiler = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_ALQUILER)
if not isinstance(df_alquiler, pd.DataFrame):
        df_alquiler = pd.read_parquet(io.BytesIO(df_alquiler))

In [ ]:
print(df_venta.columns)
print(df_alquiler.columns)

In [ ]:
df_venta["Precio/Superficie"] = df_venta["Precio"] / df_venta["Superficie"]
df_alquiler["Precio/Superficie"] = df_alquiler["Precio"] / df_alquiler["Superficie"]


In [ ]:
def resumen_estadistico(df, nombre):
    print(f"\n===== {nombre.upper()} =====")
    print(df[["Precio", "Superficie", "Precio/Superficie"]].describe())
    
    print("\nAsimetría:")
    print(df[["Precio", "Superficie", "Precio/Superficie"]].skew())
    
    print("\nCurtosis:")
    print(df[["Precio", "Superficie", "Precio/Superficie"]].kurt())

resumen_estadistico(df_venta, "ventas")
resumen_estadistico(df_alquiler, "alquiler")

Claramente podemos obervar una alta cantidad de outliers hacia la derecha, con una alta asimetría y curtosis, así como con una mediana mucho más representativa que la media.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))

sns.histplot(df_venta["Precio/Superficie"], kde=True, ax=axes[0])
axes[0].set_title("Distribución Precio/m2 - Venta")

sns.histplot(df_alquiler["Precio/Superficie"], kde=True, ax=axes[1])
axes[1].set_title("Distribución Precio/m2 - Alquiler")

plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))

sns.histplot(df_venta["Precio"], kde=True, ax=axes[0])
axes[0].set_title("Distribución Precio - Venta")

sns.histplot(df_alquiler["Precio"], kde=True, ax=axes[1])
axes[1].set_title("Distribución Precio - Alquiler")

plt.show()

Vemos que los precios tienden a la normalidad, si bien muy hacia la izquierda debido a los previamente mencionados outliers

In [ ]:
top_barrios = df_venta["Distrito"].value_counts().index

plt.figure(figsize=(14,6))
sns.boxplot(data=df_venta[df_venta["Distrito"].isin(top_barrios)],
            x="Distrito", y="Precio")

plt.xticks(rotation=45)
plt.title("Precio Venta por Distrito")
plt.show()

In [ ]:
top_barrios = df_venta["Distrito"].value_counts().index

plt.figure(figsize=(14,6))
sns.boxplot(data=df_venta[df_venta["Distrito"].isin(top_barrios)],
            x="Distrito", y="Precio/Superficie")

plt.xticks(rotation=45)
plt.title("Precio Venta por Distrito")
plt.show()

In [ ]:
top_barrios = df_alquiler["Distrito"].value_counts().index

plt.figure(figsize=(14,6))
sns.boxplot(data=df_alquiler[df_alquiler["Distrito"].isin(top_barrios)],
            x="Distrito", y="Precio")

plt.xticks(rotation=45)
plt.title("Precio Alquiler por Distrito")
plt.show()

In [ ]:
top_barrios = df_alquiler["Distrito"].value_counts().index

plt.figure(figsize=(14,6))
sns.boxplot(data=df_alquiler[df_alquiler["Distrito"].isin(top_barrios)],
            x="Distrito", y="Precio/Superficie")

plt.xticks(rotation=45)
plt.title("Precio Alquiler por Distrito")
plt.show()

Aquí tenemos una visualización más clara de los outliers, así como de los distritos de Madrid que claramente presentan precios más altos y más bajos

In [ ]:
corr_ventas = df_venta.corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr_ventas, annot=True, cmap="coolwarm", center=0)
plt.title("Matriz de Correlación - Ventas")
plt.show()

In [ ]:
corr_alquiler = df_alquiler.corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr_alquiler, annot=True, cmap="coolwarm", center=0)
plt.title("Matriz de Correlación - Alquiler")
plt.show()

Aquí la correlación más clara se da entre el precio, la superficie, el número de habitaciones y los baños. También vemos correlación mínima en caso de tener ascensor o balcón, aunque esto también puede ser debido a que a mayor tamaño de la vivienda, mayor probabilidad de que tenga múltiples pisos.

In [ ]:
sns.scatterplot(data=df_venta, x="Superficie", y="Precio")
plt.title("Precio vs m2 (Venta)")
plt.show()

sns.scatterplot(data=df_alquiler, x="Superficie", y="Precio")
plt.title("Precio vs m2 (Alquiler)")
plt.show()

In [ ]:
tabla = df_venta.pivot_table(values="Precio/Superficie",
                            index="Distrito",
                            columns="Num_habitaciones",
                            aggfunc="mean")

plt.figure(figsize=(12,8))
sns.heatmap(tabla, cmap="viridis")
plt.title("Precio/m2 por Barrio y Habitaciones")
plt.show()

In [ ]:
tabla = df_alquiler.pivot_table(values="Precio/Superficie",
                            index="Distrito",
                            columns="Num_habitaciones",
                            aggfunc="mean")

plt.figure(figsize=(12,8))
sns.heatmap(tabla, cmap="viridis")
plt.title("Precio/m2 por Barrio y Habitaciones")
plt.show()

Aquí podemos ver el precio por casa por distrito según su número de habitaciones. Claramente vemos en que barrios las casas tienen más habitaciones y dónde son más caras

In [ ]:
Q1 = df_venta["Precio/Superficie"].quantile(0.25)
Q3 = df_venta["Precio/Superficie"].quantile(0.75)
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR

outliers = df_venta[df_venta["Precio/Superficie"] > limite_superior]

print("Número de outliers:", len(outliers))

In [ ]:
Q1 = df_alquiler["Precio/Superficie"].quantile(0.25)
Q3 = df_alquiler["Precio/Superficie"].quantile(0.75)
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR

outliers = df_alquiler[df_alquiler["Precio/Superficie"] > limite_superior]

print("Número de outliers:", len(outliers))

In [ ]:
stat, p = stats.shapiro(df_venta["Precio/Superficie"].sample(500))

print("Shapiro-Wilk p-value:", p)

In [ ]:
stat, p = stats.shapiro(df_alquiler["Precio/Superficie"].sample(500))

print("Shapiro-Wilk p-value:", p)